In [7]:
import torch
import tomlkit
from lib.utils import generate_sample, pad_collate_fn, load_configs, get_data_loader, create_model
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

In [8]:
model_path = "configs/checkpoints/base_transformer_6_hour_run"

with open(model_path + "/model.toml", "rb") as f:
        model_cfg = tomlkit.load(f)
with open(model_path + "/training.toml", "rb") as f:
        train_cfg = tomlkit.load(f)
with open(model_path + "/data.toml", "rb") as f:
        data_cfg = tomlkit.load(f)

model_data = torch.load(model_path + "/ckpt_best.pt", map_location=torch.device('cpu'))

dataset, loader, vocab_size = get_data_loader(data_cfg, train_cfg)
model = create_model(model_cfg, vocab_size, device, dataset)
model.load_state_dict(model_data["model"])

NonExistentKey: 'Key "activation_kwargs" does not exist.'

In [18]:
generate_sample(model, dataset, device, "Sara and Ben are playing in the snow. They make a big snowman with a hat and a scarf. They are happy and laugh. But then a big dog comes. The dog is angry and barks. He runs to the snowman and bites his hat. Sara and Ben are scared and cry. ”Go away, dog! Leave our snowman alone!” Sara shouts. But the dog does not listen. He bites the scarf and the snowman’s nose. He shakes his head and makes the snowman fall. Sara and ", n_words=1000, max_new_tokens=500, temperature=0.1)

'sara and ben are playing in the snow. they make a big snowman with a hat and a scarf. they are happy and laugh. but then a big dog comes. the dog is angry and barks. he runs to the snowman and bites his hat. sara and ben are scared and cry. ”go away, dog! leave our snowman alone!” sara shouts. but the dog does not listen. he bites the scarf and the snowman’s nose. he shakes his head and makes the snowman fall. sara and the dog run away. the dog runs to the snowman, and they both laugh. they sit in the snow, happy and full of joy.'

In [42]:
sum(p.numel() for p in model.parameters() if p.requires_grad)

31418112

In [9]:
print(tomlkit.dumps(model_cfg))

# Model Parameters

model = "Transformer"

[global]
num_layers = 15
embedding_dim = 768
feedforward_dim = 3072
max_context = 512
activation = "GELU"
vocab_size = 4000
padding_idx = 3

[attention]
n_heads = 6
project_kv = false
projection_dim = 256
qk_dim = 128
v_dim = 128
causal = true
sdpa = true


[norm]
type = "RMSNorm"

[dropout]
attn_dropout = 0.0
ffn_dropout = 0.0

[rope]
base = 10000

